# Session 8 Homework · Is My House-Price Model Any Good?

**Expected time:** 25–35 minutes · **Bring questions** — next session opens with homework review.

This is the **first homework under the new rule: _numbers, not vibes._** You'll judge the housing price model properly, on a held-out test set, with named metrics.

**Done means:**

- test **MAE, RMSE, and R²** all reported
- a **guess-the-average baseline** the model beats on all three
- a **train vs test** comparison (did it generalise?)
- a written **verdict** citing at least one named metric *and* a reason from the price scale or baseline

*A verdict without a number does not pass.*

## Step 1 · Prep, split, fit

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

homes = pd.read_csv("../../../datasets/secondary/housing.csv")

# Fill the two missing-value columns (Session 2).
homes["age_years"] = homes["age_years"].fillna(homes["age_years"].median())
homes["distance_to_center_km"] = homes["distance_to_center_km"].fillna(homes["distance_to_center_km"].median())

features = ["area_sqft", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]
X = homes[features]
y = homes["price_lakhs"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
model = LinearRegression().fit(X_train, y_train)
pred = model.predict(X_test)
print("trained on", len(X_train), "homes; testing on", len(X_test))

## Step 2 · Report the three metrics (on the test set)

In [ ]:
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print(f"MAE:  {mae:.1f} lakhs   (typical miss)")
print(f"RMSE: {rmse:.1f} lakhs   (typical miss, big ones count more)")
print(f"R2:   {r2:.3f}          (fraction of price variation explained)")

## Step 3 · Beat the baseline?

In [ ]:
baseline = np.full(len(y_test), y_train.mean())
print(f"guess-the-average:  RMSE {np.sqrt(mean_squared_error(y_test, baseline)):.1f}   R2 {r2_score(y_test, baseline):+.2f}")
print(f"our model:          RMSE {rmse:.1f}   R2 {r2:+.2f}")
print(f"\nprices in the data run from {y.min():.0f} to {y.max():.0f} lakhs")

## Step 4 · Did it generalise? (train vs test)

In [ ]:
for name, Xs, ys in [("TRAIN", X_train, y_train), ("TEST ", X_test, y_test)]:
    p = model.predict(Xs)
    print(f"{name}  RMSE {np.sqrt(mean_squared_error(ys, p)):.1f}   R2 {r2_score(ys, p):.3f}")

### ✏️ Write your verdict

One short paragraph: **Is this a good model for pricing houses?** Use the numbers — the price scale (~18–231 lakh), the baseline, and the train/test gap. No vibes.

*Your verdict:*

This model explains about 69% of the variation in house prices (R² ≈ 0.69) and is typically off by around ₹18–22 lakh (MAE ≈ 18, RMSE ≈ 22). That's far better than guessing the average (which is off by ~₹35+ lakh, R² ≈ 0), and the train and test numbers are close, so it generalised rather than memorised. But being off by ~₹20 lakh is a lot of money when a house costs ₹100 lakh — good enough to spot rough trends, not to price a specific home. *(Accept any honest verdict backed by at least one named metric and a scale/baseline reason.)*

## One last reflection

✏️ In class the student-habits model scored R² ≈ 0.89 and was off by ~5 points; here the housing model scores R² ≈ 0.69 and is off by ~₹20 lakh. **Same workflow, different result.** In one or two sentences: what does that tell you about the phrase "good model"?

*Your answer:* "Good" isn't a property of the workflow — it depends on the problem. The identical steps produced a strong model for test scores and a rougher one for house prices, because price is harder to predict from these features. Only the named metrics let me tell the two apart — which is exactly why 'numbers, not vibes' is now the rule.